# Занятие 2. Загрузка данных: bronze и silver

Розничная сеть каждый час присылает CSV-файл с операциями касс. Описание данных —
[docs/retail-data.md](../docs/retail-data.md).

План:

1. **Bronze** — что загрузил DAG `retail_sales_bronze`: файлы как есть, плюс служебные колонки.
   Смотрим снапшоты Iceberg и путешествуем во времени.
2. **Проблемы источника** — дубли, опоздавшие строки, битые записи, отмены.
3. **Silver** — чистим данные и применяем отмены командой `MERGE INTO` (паттерн **Merger**).
   В ячейках с пометкой `TODO` нужно дописать код.

**Перед началом** (см. слайды занятия):

* данные сгенерированы: `python generator/generate_retail.py`;
* DAG `retail_dims` выполнен, DAG `retail_sales_bronze` включён и загрузил хотя бы несколько часов.

> Кластер один на всех: пока DAG загружает файлы, этот ноутбук ждёт свободных ядер (и наоборот).
> Если ячейка «висит» — посмотрите на http://localhost:8090, кто занял кластер.

In [ ]:
import sys
sys.path.append("../host")
from spark_session import get_spark
from pyspark.sql import functions as F

spark = get_spark("lab-02")

## 1. Bronze: что загрузил DAG

Каждый запуск DAG отвечает за один час и загружает файл `sales_<час>.csv` в таблицу
`iceberg.retail.sales_bronze`. Строки кладутся **как есть**, к ним добавляются три колонки:

| колонка | смысл |
|---|---|
| `file_hour` | за какой час пришёл файл (по нему таблица партиционирована) |
| `source_file` | имя файла |
| `loaded_at` | когда строку загрузили |

In [ ]:
spark.sql("SHOW TABLES IN iceberg.retail").show()
spark.table("iceberg.retail.sales_bronze").printSchema()

In [ ]:
# Сколько строк пришло в каждом файле
spark.sql("""
    SELECT file_hour, source_file, COUNT(*) AS rows
    FROM iceberg.retail.sales_bronze
    GROUP BY file_hour, source_file
    ORDER BY file_hour
""").show(80, truncate=False)

In [ ]:
spark.table("iceberg.retail.sales_bronze").limit(10).toPandas()

### Снапшоты: каждая загрузка — отдельная версия таблицы

Iceberg хранит историю изменений в служебных таблицах: `<таблица>.snapshots`, `.history`,
`.files`, `.partitions`. Каждый запуск DAG добавил снапшот с операцией `overwrite`: загрузка
заменяет партицию своего часа целиком (паттерн **Data Overwrite**). Поэтому перезапуск DAG
за тот же час не удваивает данные.

In [ ]:
spark.sql("""
    SELECT committed_at, snapshot_id, operation,
           summary['added-records']   AS added,
           summary['deleted-records'] AS deleted,
           summary['total-records']   AS total
    FROM iceberg.retail.sales_bronze.snapshots
    ORDER BY committed_at
""").show(80, truncate=False)

In [ ]:
# Путешествие во времени: таблица такой, какой она была после первой загрузки
first = spark.sql(
    "SELECT snapshot_id FROM iceberg.retail.sales_bronze.snapshots ORDER BY committed_at LIMIT 1"
).first()["snapshot_id"]

spark.sql(f"SELECT COUNT(*) AS rows FROM iceberg.retail.sales_bronze VERSION AS OF {first}").show()
spark.sql("SELECT COUNT(*) AS rows FROM iceberg.retail.sales_bronze").show()

**Попробуйте:** в Airflow откройте любой успешный запуск `retail_sales_bronze` → задача
`load_hour` → **Clear**. Airflow выполнит загрузку этого часа ещё раз. Затем перезапустите
две ячейки выше: появится новый снапшот, а число строк в таблице не изменится.

## 2. Проблемы источника

Прежде чем чистить данные, найдём, что с ними не так. Ниже готовые запросы — выполните их
и запишите для себя ответы: сколько каких проблем в данных.

In [ ]:
# Типы операций
spark.sql("""
    SELECT operation_type, COUNT(*) AS rows
    FROM iceberg.retail.sales_bronze GROUP BY operation_type
""").show()

In [ ]:
# Дубли: одна и та же операция (event_id) встречается несколько раз
spark.sql("""
    SELECT COUNT(*) AS rows, COUNT(DISTINCT event_id) AS unique_events,
           COUNT(*) - COUNT(DISTINCT event_id) AS duplicates
    FROM iceberg.retail.sales_bronze
""").show()

# ...в том числе в разных файлах: источник переотправил часть предыдущего часа
spark.sql("""
    SELECT COUNT(*) AS events_in_several_files FROM (
        SELECT event_id FROM iceberg.retail.sales_bronze
        GROUP BY event_id HAVING COUNT(DISTINCT source_file) > 1)
""").show()

In [ ]:
# Опоздавшие строки: операция случилась раньше часа файла (офлайн-касса выгрузила чеки позже)
spark.sql("""
    SELECT (unix_timestamp(file_hour) - unix_timestamp(date_trunc('HOUR', event_ts))) / 3600 AS late_hours,
           COUNT(*) AS rows
    FROM iceberg.retail.sales_bronze
    GROUP BY 1 ORDER BY 1
""").show()

In [ ]:
# Битые строки
spark.sql("""
    SELECT
        SUM(CASE WHEN price_paid_kop < 0 THEN 1 ELSE 0 END)            AS negative_price,
        SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END)            AS no_product,
        SUM(CASE WHEN store_id NOT IN (SELECT store_id FROM iceberg.retail.stores)
                 THEN 1 ELSE 0 END)                                     AS unknown_store
    FROM iceberg.retail.sales_bronze
""").show()

In [ ]:
# Отмена ссылается на продажу через ref_event_id: та же позиция того же чека
cancel = spark.sql(
    "SELECT * FROM iceberg.retail.sales_bronze WHERE operation_type = 'CANCEL' LIMIT 1").first()
spark.sql(f"""
    SELECT event_id, event_ts, receipt_id, line_no, product_id, operation_type,
           price_paid_kop, ref_event_id, source_file
    FROM iceberg.retail.sales_bronze
    WHERE event_id IN ({cancel.event_id}, {cancel.ref_event_id})
    ORDER BY event_ts
""").show(truncate=False)

## 3. Silver: чистые данные

Таблица `iceberg.retail.sales_silver` — то, на чём строится аналитика:

* без битых строк и без дублей: **одна строка = одна операция** (`event_id` уникален);
* только продажи (`SALE`) и возвраты (`RETURN`). Отмены (`CANCEL`) не хранятся отдельными
  строками — они выставляют у отменённой продажи флаг `is_cancelled = true`;
* колонка `event_date` — дата операции, по ней таблица партиционирована.

### Паттерн Merger

Bronze постоянно пополняется, и silver нужно обновлять снова и снова. Если просто
дописывать (`append`) новые строки, повторный запуск или переотправленный источником файл
создадут дубли. Паттерн **Merger** (Data Engineering Design Patterns, гл. 4) решает это так:
новые данные **сливаются** с таблицей по ключу командой `MERGE INTO`:

```sql
MERGE INTO <целевая таблица> t
USING <новые данные> s
ON <условие совпадения по ключу>
WHEN MATCHED THEN UPDATE ...        -- ключ уже есть: обновить строку (или ничего не делать)
WHEN NOT MATCHED THEN INSERT ...    -- ключа ещё нет: вставить
```

Сколько раз ни запускай такой `MERGE` на одних и тех же данных, результат один и тот же.
Это свойство называется **идемпотентностью**.

In [ ]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS iceberg.retail.sales_silver (
        event_id BIGINT, event_ts TIMESTAMP, receipt_id BIGINT, line_no INT,
        store_id INT, product_id INT, category_id INT, customer_id BIGINT,
        is_loyalty BOOLEAN, operation_type STRING, quantity INT,
        price_regular_kop BIGINT, price_paid_kop BIGINT, payment_method STRING,
        ref_event_id BIGINT,
        is_cancelled BOOLEAN,
        event_date DATE
    ) USING iceberg
    PARTITIONED BY (event_date)
""")

### Шаг 1. Убираем битые строки

Корректная строка: цена не отрицательная, товар указан, магазин есть в справочнике.

In [ ]:
bronze = spark.table("iceberg.retail.sales_bronze")
known_stores = spark.table("iceberg.retail.stores").select("store_id")

# TODO 1: оставьте в valid только корректные строки bronze:
#   * price_paid_kop >= 0
#   * product_id не пустой (подсказка: F.col(...).isNotNull())
#   * store_id есть в справочнике магазинов
#     (подсказка: .join(known_stores, "store_id", "left_semi") оставляет строки,
#      для которых нашлась пара, и не добавляет колонок)
valid = bronze  # ← замените

print("bronze:", bronze.count(), " корректных:", valid.count())

### Шаг 2. Убираем дубли

Дубли — точные копии одной операции, у них одинаковый `event_id`. Оставляем по одной строке
на каждый `event_id`.

In [ ]:
# TODO 2: оставьте по одной строке на каждый event_id (подсказка: dropDuplicates)
deduped = valid  # ← замените

batch = (
    deduped
    .withColumn("event_date", F.to_date("event_ts"))
    .withColumn("is_cancelled", F.lit(False))
    .drop("file_hour", "source_file", "loaded_at")
)
# Продажи и возвраты становятся строками silver, отмены — обновлениями
batch.where(F.col("operation_type") != "CANCEL").createOrReplaceTempView("batch_events")
# ref_event_id переименован: с одноимённой колонкой silver MERGE в Spark 3.5 падает
batch.where(F.col("operation_type") == "CANCEL") \
     .select(F.col("ref_event_id").alias("sale_event_id")).distinct() \
     .createOrReplaceTempView("batch_cancels")

print("уникальных операций:", deduped.count())

### Шаг 3. MERGE: добавляем новые операции

Строка из `batch_events` вставляется, только если такой операции в silver ещё нет.
Условие `t.event_date = s.event_date` не меняет результат (у операции одна дата), но
позволяет Iceberg читать только партиции нужных дней, а не всю таблицу.

In [ ]:
# TODO 3: допишите условие ON — операция уже есть в silver, если совпадает event_id
spark.sql("""
    MERGE INTO iceberg.retail.sales_silver t
    USING batch_events s
    ON ... AND t.event_date = s.event_date
    WHEN NOT MATCHED THEN INSERT *
""")

### Шаг 4. MERGE: применяем отмены

Отмена `CANCEL` ссылается на продажу через `ref_event_id`. Найденная продажа должна
получить `is_cancelled = true`. Отмены уже применённых продаж пропускаем
(`AND NOT t.is_cancelled`): так повторный запуск не переписывает файлы зря.

In [ ]:
# TODO 4: допишите действие для найденной продажи — выставить флаг отмены
spark.sql("""
    MERGE INTO iceberg.retail.sales_silver t
    USING batch_cancels c
    ON t.event_id = c.sale_event_id
    WHEN MATCHED AND NOT t.is_cancelled THEN UPDATE SET ...
""")

### Проверка

Все строки ниже должны вывести `OK`.

In [ ]:
silver = spark.table("iceberg.retail.sales_silver")
check = spark.sql("""
    SELECT COUNT(*) AS rows, COUNT(DISTINCT event_id) AS unique_events,
           SUM(CASE WHEN operation_type = 'CANCEL' THEN 1 ELSE 0 END) AS cancel_rows,
           SUM(CASE WHEN is_cancelled THEN 1 ELSE 0 END) AS cancelled_sales,
           SUM(CASE WHEN price_paid_kop < 0 OR product_id IS NULL THEN 1 ELSE 0 END) AS broken
    FROM iceberg.retail.sales_silver
""").first()
print(check)
print("дублей нет:       ", "OK" if check.rows == check.unique_events else "ОШИБКА")
print("отмен-строк нет:  ", "OK" if check.cancel_rows == 0 else "ОШИБКА")
print("отмены применены: ", "OK" if check.cancelled_sales > 0 else "ОШИБКА")
print("битых строк нет:  ", "OK" if check.broken == 0 else "ОШИБКА")

### Идемпотентность

Выполните ячейки шагов 1–4 **ещё раз** и снова запустите проверку: число строк не изменится.
В истории снапшотов silver видно, что повторные `MERGE` ничего не добавили.

Если DAG тем временем загрузил новые часы, повторный запуск добавит **только** новые
операции и применит новые отмены. Так silver обновляется инкрементально.

In [ ]:
spark.sql("""
    SELECT committed_at, operation,
           summary['added-records']   AS added,
           summary['deleted-records'] AS deleted,
           summary['total-records']   AS total
    FROM iceberg.retail.sales_silver.snapshots
    ORDER BY committed_at
""").show(truncate=False)

Отметьте в выводе: `MERGE` с отменами — это операция `overwrite`, у которой `deleted`
не ноль. Файлы в HDFS не изменяются на месте (занятие 1), поэтому Iceberg переписывает
файлы, в которых есть отменённые продажи, и публикует новый снапшот.

## Итоги

* **Bronze** хранит файлы как есть. Загрузка перезаписывает партицию своего часа
  (**Data Overwrite**), поэтому её можно безопасно повторять.
* **Silver** — одна строка на операцию, отмены учтены. `MERGE INTO` по ключу
  (**Merger**) делает обновление идемпотентным.
* Каждое изменение таблицы — снапшот. История и путешествие во времени бесплатны.

Не забудьте остановить сессию — иначе кластер занят и DAG стоит в очереди.

In [ ]:
spark.stop()